In [2]:
import pip
!pip install pmdarima

   ---------------------------------------- 0.0/715.6 kB ? eta -:--:--
   ---------------------------------------- 0.0/715.6 kB ? eta -:--:--
   -------------- ------------------------- 262.1/715.6 kB ? eta -:--:--
   ---------------------------------------- 715.6/715.6 kB 1.8 MB/s  0:00:00
   ---------------------------------------- 0.0/2.8 MB ? eta -:--:--
   --- ------------------------------------ 0.3/2.8 MB ? eta -:--:--
   ------- -------------------------------- 0.5/2.8 MB 1.7 MB/s eta 0:00:02
   ----------- ---------------------------- 0.8/2.8 MB 1.7 MB/s eta 0:00:02
   --------------- ------------------------ 1.0/2.8 MB 1.9 MB/s eta 0:00:01
   ---------------------- ----------------- 1.6/2.8 MB 1.6 MB/s eta 0:00:01
   -------------------------- ------------- 1.8/2.8 MB 1.7 MB/s eta 0:00:01
   ------------------------------ --------- 2.1/2.8 MB 1.6 MB/s eta 0:00:01
   ---------------------------------- ----- 2.4/2.8 MB 1.6 MB/s eta 0:00:01
   -----------------------------------


[notice] A new release of pip is available: 25.1.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
!pip install tensorflow

  Using cached tensorflow-2.21.0-cp312-cp312-win_amd64.whl.metadata (4.5 kB)
  Using cached absl_py-2.5.0-py3-none-any.whl.metadata (3.3 kB)
  Using cached astunparse-1.6.3-py2.py3-none-any.whl.metadata (4.4 kB)
  Using cached flatbuffers-25.12.19-py2.py3-none-any.whl.metadata (1.0 kB)
  Using cached gast-0.7.0-py3-none-any.whl.metadata (1.5 kB)
  Using cached google_pasta-0.2.0-py3-none-any.whl.metadata (814 bytes)
  Using cached libclang-18.1.1-py2.py3-none-win_amd64.whl.metadata (5.3 kB)
  Using cached opt_einsum-3.4.0-py3-none-any.whl.metadata (6.3 kB)
  Using cached protobuf-7.35.1-cp310-abi3-win_amd64.whl.metadata (595 bytes)
  Using cached termcolor-3.3.0-py3-none-any.whl.metadata (6.5 kB)
  Using cached grpcio-1.82.1-cp312-cp312-win_amd64.whl.metadata (3.8 kB)
  Using cached keras-3.15.0-py3-none-any.whl.metadata (6.3 kB)
  Using cached ml_dtypes-0.5.4-cp312-cp312-win_amd64.whl.metadata (9.2 kB)
  Using cached namex-0.1.0-py3-none-any.whl.metadata (322 bytes)
  Using cached opt

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
mlflow-skinny 3.2.0 requires protobuf<7,>=3.12.0, but you have protobuf 7.35.1 which is incompatible.
mlflow-tracing 3.2.0 requires protobuf<7,>=3.12.0, but you have protobuf 7.35.1 which is incompatible.
streamlit 1.52.2 requires protobuf<7,>=3.20, but you have protobuf 7.35.1 which is incompatible.

[notice] A new release of pip is available: 25.1.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [10]:
!pip install xgboost

   ---------------------------------------- 0.0/69.5 MB ? eta -:--:--
   ---------------------------------------- 0.0/69.5 MB ? eta -:--:--
   ---------------------------------------- 0.3/69.5 MB ? eta -:--:--
   ---------------------------------------- 0.8/69.5 MB 2.6 MB/s eta 0:00:27
    --------------------------------------- 1.0/69.5 MB 1.9 MB/s eta 0:00:36
    --------------------------------------- 1.6/69.5 MB 2.2 MB/s eta 0:00:32
   - -------------------------------------- 2.4/69.5 MB 2.3 MB/s eta 0:00:30
   - -------------------------------------- 2.9/69.5 MB 2.3 MB/s eta 0:00:29
   -- ------------------------------------- 3.7/69.5 MB 2.5 MB/s eta 0:00:26
   -- ------------------------------------- 4.2/69.5 MB 2.6 MB/s eta 0:00:26
   -- ------------------------------------- 5.0/69.5 MB 2.7 MB/s eta 0:00:24
   --- ------------------------------------ 5.5/69.5 MB 2.7 MB/s eta 0:00:24
   --- ------------------------------------ 6.3/69.5 MB 2.8 MB/s eta 0:00:23
   --- -------------


[notice] A new release of pip is available: 25.1.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [11]:
# import libraries
from gettext import install
import sys
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import pip
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..')))
from src.engine import get_metrics, create_lstm_sequence
from sklearn.model_selection import TimeSeriesSplit
from sklearn.ensemble import RandomForestRegressor
import pmdarima as pm
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense
import warnings
from sqlalchemy import values
from xgboost import XGBRegressor

In [4]:
# load csv
df = pd.read_csv(r'../data/processed/cleaned_featured_data.csv', parse_dates=['observation_date'], index_col='observation_date')
df_clean = df.dropna()

In [5]:
# train test split
tscv = TimeSeriesSplit(n_splits=5)
train_size = int(len(df_clean) * 0.8)
train, test = df_clean.iloc[:train_size], df_clean.iloc[train_size:]

print(f"Train size: {len(train)}, Test size: {len(test)}")
print(f"Testing shape: {test.shape}, Training shape: {train.shape}")

Train size: 1020, Test size: 255
Testing shape: (255, 6), Training shape: (1020, 6)


In [7]:
# SARIMA Model
print("Fitting SARIMA model...")
sarima_model = pm.auto_arima(train['DHHNGSP'], seasonal=True, m=7, suppress_warnings=True)

# Forecasting with SARIMA
sarima_preds, sarima_conf_int = sarima_model.predict(n_periods=len(test), return_conf_int=True)
sarima_metrics = get_metrics(test['DHHNGSP']. values, sarima_preds.values)
print(f"SARIMA Metrics: {sarima_metrics}")

Fitting SARIMA model...
SARIMA Metrics: {'MAE': 0.7988613546566415, 'RMSE': 2.51389782494208, 'MAPE': 16.141396118070293}


In [8]:
# Random Forest Model
print("Fitting Random Forest model...")
features = ['lag_1', 'lag_2', 'lag_3', 'Rolling_Std_7']
X_train, y_train = train[features], train['DHHNGSP']
X_test, y_test = test[features], test['DHHNGSP']

rf_model = RandomForestRegressor(n_estimators=100, random_state=42)
rf_model.fit(X_train, y_train)

# Forecasting with Random Forest
rf_preds = rf_model.predict(X_test)
rf_metrics = get_metrics(y_test.values, rf_preds)
print(f"Random Forest Metrics: {rf_metrics}")

Fitting Random Forest model...
Random Forest Metrics: {'MAE': 0.4223030392156868, 'RMSE': 2.060897350072722, 'MAPE': 6.1882752539184995}


In [12]:
print("Fitting XGBoost model...")
xgb_model = XGBRegressor(n_estimators=100, random_state=42, learning_rate=0.1)
xgb_model.fit(X_train, y_train)

# Forecasting with XGBoost
xgb_preds = xgb_model.predict(X_test)
xgb_metrics = get_metrics(y_test.values, xgb_preds)
print(f"XGBoost Metrics: {xgb_metrics}")

Fitting XGBoost model...
XGBoost Metrics: {'MAE': 0.4386604853985356, 'RMSE': 2.165315962971091, 'MAPE': 6.272163738619395}


In [13]:
# LSTM Model
print("Fitting LSTM model...")
target_series = df_clean['DHHNGSP'].values
time_steps = 30

X_lstm, y_lstm = create_lstm_sequence(target_series, time_steps)

# Choronological split for LSTM
lstm_train_size = int(len(X_lstm) * 0.8)
X_lstm_train, X_lstm_test = X_lstm[:lstm_train_size], X_lstm[lstm_train_size:]
y_lstm_train, y_lstm_test = y_lstm[:lstm_train_size], y_lstm[lstm_train_size:]

# Reshape X to (samples, time_steps, features)
X_lstm_train = X_lstm_train.reshape((X_lstm_train.shape[0], X_lstm_train.shape[1], 1))
X_lstm_test = X_lstm_test.reshape((X_lstm_test.shape[0], X_lstm_test.shape[1], 1))

# Build LSTM model
lstm_model = Sequential([
    LSTM(50, activation='relu', input_shape=(time_steps, 1)),
    Dense(1)
])
lstm_model.compile(optimizer='adam', loss='mse')

# fIT the model
lstm_model.fit(X_lstm_train, y_lstm_train, epochs=20, batch_size=32, verbose=1)

# Forecasting with LSTM
lstm_preds = lstm_model.predict(X_lstm_test, verbose=0).flatten()
lstm_metrics = get_metrics(y_lstm_test, lstm_preds)
print(f"LSTM Metrics: {lstm_metrics}")

Fitting LSTM model...


c:\Users\User\anaconda3\Lib\site-packages\keras\src\layers\rnn\rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Epoch 1/20
32/32 ━━━━━━━━━━━━━━━━━━━━ 2s 11ms/step - loss: 9.1100
Epoch 2/20
32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - loss: 0.7234
Epoch 3/20
32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - loss: 0.3862
Epoch 4/20
32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - loss: 0.3534
Epoch 5/20
32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - loss: 0.3351
Epoch 6/20
32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - loss: 0.3060
Epoch 7/20
32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - loss: 0.2868
Epoch 8/20
32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - loss: 0.2634
Epoch 9/20
32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - loss: 0.2580
Epoch 10/20
32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - loss: 0.2546
Epoch 11/20
32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - loss: 0.2470
Epoch 12/20
32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - loss: 0.2439
Epoch 13/20
32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - loss: 0.2383
Epoch 14/20
32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - loss: 0.2608
Epoch 15/20
32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - loss: 0.2478
Epoch 1